## Figures 3-9 - Reduced-order model of pleural pressure distribution

In [ ]:
# env: notebook
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
import pickle

from collections import Counter, defaultdict
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from matplotlib.ticker import MaxNLocator
from matplotlib.lines import Line2D

import dolfin
import torch

from Reader_MeshDisplacement import MeshDisplacementReader
from funtd import FTD
from TensorDecomposition_CP import CP
from Residual_Sampling import ResidualSampling
from Figures_FTD import FiguresFTD

In [ ]:
# %matplotlib widget
%load_ext autoreload
%autoreload 2

In [ ]:
def symm_cmap_and_norm(values):
    # colors = ['#084594', 'white', '#cb181d']
    colors = ["tab:blue", 'white', 'tab:red']
    cmap = LinearSegmentedColormap.from_list("blue_white_red", colors, N=256)

    # Define symmetric normalization (centered at 0)
    vmin, vcenter, vmax = values.min(), 0, values.max()
    vmax_abs = max(abs(vmin), abs(vmax))  # makes color intensity symmetric
    norm = TwoSlopeNorm(vmin=-vmax_abs, vcenter=vcenter, vmax=vmax_abs)

    return cmap, norm

### Download data

In [ ]:
# TODO Update links to zenodo

In [ ]:
!curl -L -o results_reduced_model.zip "https://sdrive.cnrs.fr/s/bMBFcWQF4QmSqDD/download?path=%2F&files=Results_reduced_model"
!unzip -o results_reduced_model.zip
!rm -f results_reduced_model.zip

In [ ]:
# TODO : Download also excel data file

### Parameters

In [ ]:
pe          = -0.5    # kPa
alpha_lst   = [0.16]
gravity_lst = [1] # [0,1]

rho_solid   = 1e-6
g           = +9.81e3 # mm/s2

### Get pressure data

In [ ]:
region_lst = []
region_lst += ["LL"]
region_lst += ["RL"]

n_phases = 32

In [ ]:
Simulations_name = []
Simulations_name.append("alpha")
Simulations_name.append(alpha_lst)
Simulations_name.append("pe")
Simulations_name.append(pe)

flat_Simulations_name = [item for sublist in Simulations_name for item in (sublist if isinstance(sublist, list) else [sublist])]
Simulations_filename  = '_'.join([str(x) for x in flat_Simulations_name])

# Load the pressure data for the construction of the reduced-order model. It can also be generated in the notebook Fig2_save_transported_fields.ipynb
with open(f"Results_reduced_model/{Simulations_filename}/results_volunteers.pkl", "rb") as f:
    results_volunteers = pickle.load(f)

In [ ]:
df_info = pd.read_excel("./CRF_gb_spiro3d.xlsx", header=[2])
df_info = df_info.iloc[1:, 1:]

In [ ]:
# Concatenate data from both lungs
grouped_data = defaultdict(lambda: defaultdict(list))

for sim in results_volunteers.values():
    acq, vol, reg = sim["acquisition"], sim["ID"], sim["region"]
    p_vol         = sim["pf_transported"] # / np.linalg.norm(sim["p_transported"])
    volume_vol    = sim["volume_phases"]

    grouped_data[(acq, vol)][reg].append(p_vol)
    grouped_data[(acq, vol)][f"{reg}_volume"].append(volume_vol)

reduced_model = defaultdict(dict)

for (acq, vol), regions_dict in grouped_data.items():
    # Identify which regions are missing for this specific volunteer/position
    missing = [r for r in region_lst if r not in regions_dict]
    
    if not missing: # All regions present
        p_pat_lst = []
        volume_vol_reg = np.zeros((1,n_phases))
        
        for reg in region_lst:
            p_pat_lst.extend(regions_dict[reg])
            volume_vol_reg += np.array(regions_dict[f"{reg}_volume"])
        
        if acq == "ANTE_PRO1" and vol == "251128_02LS24":
            # NOTE The mesh of the LL in volunteer 251128_02LS24 in ANTE_PRO1 around phases 25, 26 is not correct (in the dynamic mask, some of the right lung, at the base, is defined as left lung). 
            # However, this does not affect the results of the pleural pressure, so we can include them. 
            # To fix this, I manually compute the correct DeltaV_V.
            vol_max  = np.max(volume_vol_reg)
            vol_min  = np.min((volume_vol_reg[0,0], volume_vol_reg[0,-1]))
            DeltaV_V = (vol_max - vol_min) / (volume_vol_reg[0,0])
        else:
            DeltaV_V = (np.max(volume_vol_reg)-np.min(volume_vol_reg))/(volume_vol_reg[0,0])

        if DeltaV_V < 0.5: # In case we want to filter some volunteers with high volume variation
            # If the criteria is DeltaV_V < 0.25, many volunteers with asthma are excluded.
            p_concatenated = np.concatenate(p_pat_lst, axis=1)
            norm           = np.linalg.norm(p_concatenated)

            reduced_model[(acq, vol)]["p_transported_concatenated"]      = p_concatenated #/ norm
            reduced_model[(acq, vol)]["p_transported_concatenated_norm"] = norm
            reduced_model[(acq, vol)]["p_transported_concatenated_max"]  = np.max(p_concatenated)

            # reduced_model[(acq, vol)]["p_end_inh_norm"] = np.mean(np.linalg.norm(p_concatenated[15:17,:], axis=1))
            reduced_model[(acq, vol)]["p_end_inh_mean"] = np.mean(np.mean(p_concatenated[15:17,:], axis=1))

            reduced_model[(acq, vol)]["volume_exhalation"] = volume_vol_reg[0,0]
            reduced_model[(acq, vol)]["Delta_volume"]      = np.max(volume_vol_reg) - np.min(volume_vol_reg)
            reduced_model[(acq, vol)]["DeltaV_V"]          = DeltaV_V

            if acq in ["ANTE_SUP1", "ANTE_SUP2", "POST_SUP1"]:
                position = "supine"
            elif acq in ["ANTE_PRO1", "ANTE_PRO2", "POST_PRO1"]:
                position = "prone"
            else:
                raise ValueError(f"Unknown position for acquisition {acq}")
            
            reduced_model[(acq, vol)]["position"] = position
            
        else:
            print(f"Skipping volunteer {vol} in {acq} due to high DeltaV_V: {100.*DeltaV_V}%")
            continue
    
    else:
        print (f"Skipping volunteer {vol} in {acq} due to missing regions: {missing}")

n_reduced_model = len(reduced_model)

P_transported = []
for red_mod in reduced_model.values():
    P_transported.append(red_mod["p_transported_concatenated"])

In [ ]:
count_ante_sup2 = sum(1 for key in reduced_model.keys() if key[0] == "ANTE_SUP2")
count_ante_pro1 = sum(1 for key in reduced_model.keys() if key[0] == "ANTE_PRO1")

print (f"Total number of reduced model entries: {n_reduced_model}")
print(f"Number of 'ANTE_SUP2' elements: {count_ante_sup2}")
print(f"Number of 'ANTE_PRO1' elements: {count_ante_pro1}")

In [ ]:
sup2_ids = {key[1] for key in reduced_model.keys() if key[0] == "ANTE_SUP2"}
pro1_ids = {key[1] for key in reduced_model.keys() if key[0] == "ANTE_PRO1"}

both = sup2_ids.intersection(pro1_ids)
only_sup2 = sup2_ids - pro1_ids
only_pro1 = pro1_ids - sup2_ids

print(f"Present in BOTH: {len(both)}")
print(f"Only in ANTE_SUP2: {len(only_sup2)}")
print(f"Only in ANTE_PRO1: {len(only_pro1)}")

#### Target mesh

In [ ]:
vol_ID_ref      = "230704_02JY02"
acquisition_ref = "ANTE_SUP2"

for simulation_i in results_volunteers.values():
    if simulation_i["ID"] == vol_ID_ref and simulation_i["acquisition"] == acquisition_ref:
        alpha_ref   = simulation_i["alpha"]
        gravity_ref = simulation_i["gravity"]
        pe_ref      = simulation_i["pe"]
        break

resultsPath              = f"./Results_{acquisition_ref}/{vol_ID_ref}"
fs_DG0_regions_bmesh_ref = {}

for region in region_lst:
    mesh_unloaded = dolfin.Mesh()
    with dolfin.XDMFFile(f"{resultsPath}/Pleural_pressure_estimation/"
                         f"mesh_unloaded_{region}_alpha{alpha_ref}_gravity{gravity_ref}_pe{pe_ref}.xdmf") as xdmf_in:
        xdmf_in.read(mesh_unloaded)

    bmesh_unloaded = dolfin.BoundaryMesh(mesh_unloaded, "exterior")
    fs_DG0         = dolfin.FunctionSpace(bmesh_unloaded, "DG", 0)
    fs_DG0_regions_bmesh_ref[region] = fs_DG0

### Functional Tensor Decomposition

In [ ]:
os.makedirs(f"Results_reduced_model/{Simulations_filename}/FunctionalTensorDecomposition", exist_ok=True)
os.makedirs(f"Results_reduced_model/{Simulations_filename}/FunctionalTensorDecomposition/CP_reference", exist_ok=True)
os.makedirs(f"Results_reduced_model/{Simulations_filename}/FunctionalTensorDecomposition/Sampling_new_fields", exist_ok=True)

In [ ]:
biomarkers = []
biomarkers += ["ID"]
biomarkers += ["Position"]
biomarkers += ["Groupe"]
biomarkers += ["Sex"]
biomarkers += ["Age"]
biomarkers += ["Height"]
biomarkers += ["Weight"]
biomarkers += ["BMI"]
biomarkers += ["p_transported_concatenated_norm"]
biomarkers += ["volume_exhalation"]
biomarkers += ["Delta_volume"]
biomarkers += ["DeltaV_V"]
biomarkers += ["p_end_inh_mean"]

# biomarkers += ["p_transported_concatenated_max"]
# biomarkers += ["CV", "CI", "CVF", "VEMS", "VEMS/CVF", "VEMS/CV", "DEM", "Texp"]

P_transported_ftd = []
var_dict = defaultdict(list)

for (acq, vol_ID), red_mod in reduced_model.items():
    # Uncomment to restrict certain cases
    vol_state = df_info.loc[df_info['ID_Volunteer'] == vol_ID[-6:], "Groupe"].values[0]
    # if vol_state == "healthy":
    #     continue
    # if vol_state == "asthma":
    #     continue
    # if vol_state == "COPD":
    #     continue

    # if acq == "ANTE_SUP2":
    #     continue
    # if acq == "ANTE_PRO1":
    #     continue

    if vol_ID == "230728_02CL05":
        print(f"Excluding outlier: Patient {vol_ID}, to be used later for comparison with the new sampled fields")
        continue

    # exclude, as they have p_norm value much higher than the rest of the patients
    # p_norm > 300
    if (vol_ID == "250207_02SN36" and acq == "ANTE_PRO1") or (vol_ID == "241217_02JC30" and acq == "ANTE_SUP2") or\
       (vol_ID == "251128_02LS24" and acq == "ANTE_PRO1"):
        print (f"Excluding outlier: Patient {vol_ID}, {vol_state}, with acquisition {acq} due to extremely high p_transported_concatenated_norm value")
        continue
    # p_norm > 200
    if (vol_ID == "240524_02TA22" and acq == "ANTE_SUP2") or (vol_ID == "231020_02AR09" and acq == "ANTE_PRO1") or\
       (vol_ID == "240206_02HA04" and acq == "ANTE_PRO1"):
        print (f"Excluding outlier: Patient {vol_ID}, {vol_state}, with acquisition {acq} due to extremely high p_transported_concatenated_norm value")
        continue

    # Patient ID: 240524_02TA22, Pos. 0, Groupe 1, MSE: 0.525, p_transported_concatenated_norm: 216.19903901051254
    # Patient ID: 231020_02AR09, Pos. 1, Groupe 0, MSE: 0.586, p_transported_concatenated_norm: 245.36290203122573
    # Patient ID: 240206_02HA04, Pos. 1, Groupe 1, MSE: 0.265, p_transported_concatenated_norm: 209.77752326098775

    # Patient ID: 241217_02JC30, Pos. 0, Groupe 0, MSE: 0.842, p_transported_concatenated_norm: 342.0164982777061
    # Patient ID: 251128_02LS24, Pos. 1, Groupe 1, MSE: 0.905, p_transported_concatenated_norm: 869.0805591966463

    P_transported_ftd.append(red_mod["p_transported_concatenated"])

    for var in biomarkers:
        if var == "ID":
            vol_var = vol_ID

        elif var == "Position":
            vol_pos = acq
            if vol_pos == "ANTE_SUP2":
                vol_var = 0
            elif vol_pos == "ANTE_PRO1":
                vol_var = 1
            else:
                assert 0, f"Unknown position {vol_pos}"

        elif var in ["p_transported_concatenated_norm", "p_transported_concatenated_max", "volume_exhalation", "Delta_volume", "DeltaV_V", "gw_dist", "p_end_inh_mean"]:
            vol_var = reduced_model[(acq, vol_ID)][var]

        elif var == "Groupe":
            vol_state = df_info.loc[df_info['ID_Volunteer'] == vol_ID[-6:], var].values[0]
            if vol_state == "healthy":
                vol_var = 0
            elif vol_state == "asthma":
                vol_var = 1
            elif vol_state == "COPD":
                vol_var = 2
            else:
                assert 0, f"Unknown group {vol_state}"

        elif var == "Sex":
            vol_sex = df_info.loc[df_info['ID_Volunteer'] == vol_ID[-6:], var].values[0]
            if vol_sex == "M":
                vol_var = 0
            elif vol_sex == "F":
                vol_var = 1
            else:
                assert 0, f"Unknown sex {vol_sex}"
        else:
            vol_var = df_info.loc[df_info['ID_Volunteer'] == vol_ID[-6:], var].values[0]

        # if var == "Groupe"and vol_var == "COPD":
        #     continue
        
        var_dict[var].append(vol_var)

P_3d_ftd = np.stack(P_transported_ftd, axis=0)
P_3d_ftd = np.transpose(P_3d_ftd, (2, 1, 0)) # (n_points, n_phases, n_vol)
print(P_3d_ftd.shape)

In [ ]:
# FTD parameters
S, T, P = P_3d_ftd.shape   # spatial pts, time pts, patients
K = 4                      # Number of P1 nodes for the variables

R_lst = np.arange(1, 4)

# Data
# X_observed = torch.from_numpy(P_3d_ftd).float()
X_observed = P_3d_ftd.copy()

# Variables
variables_ftd_combinations = []

# variables_ftd_combinations.append(["Height"])
# variables_ftd_combinations.append(["Height"])
# variables_ftd_combinations.append(["Weight"])
# variables_ftd_combinations.append(["BMI"])
# variables_ftd_combinations.append(["p_transported_concatenated_norm"])
# variables_ftd_combinations.append(["DeltaV_V"])
# variables_ftd_combinations.append(["volume_exhalation"])
# variables_ftd_combinations.append(["Delta_volume"])
# variables_ftd_combinations.append(["DeltaV_V"])
# variables_ftd_combinations.append(["gw_dist-Groupe"])

# Variables for the article
variables_ftd_combinations.append(["Position"])
variables_ftd_combinations.append(["Groupe"])
variables_ftd_combinations.append(["Sex"])

variables_ftd_combinations.append(["Age"])
variables_ftd_combinations.append(["BMI"])
variables_ftd_combinations.append(["p_end_inh_mean"])

variables_ftd_combinations.append(["Age-Position"])
variables_ftd_combinations.append(["BMI-Position"])
variables_ftd_combinations.append(["p_end_inh_mean-Position"])

results_als_combinations = defaultdict(lambda: defaultdict(list))

mse_parafac_lst   = [] # Error of the CP decomposition for each R
for R in R_lst:
    print (f"Running Parafac + ALS with rank R={R}")
    # Parafac decomposition for initialization
    normalized_parafac, mse_parafac = CP().get_normalized_parafac_decomposition(data_tensor=P_3d_ftd, rank=R)
    mse_parafac_lst.append(mse_parafac)

    for variables_ftd_combination in variables_ftd_combinations:
        ftd = FTD(variables_ftd_combination, var_dict, K)
        Phi, nodes_dict = ftd.build_global_design_matrix()

        mse_als_train_lst = []
        mse_als_test_lst  = []

        # Alternating Least Squares (ALS)
        # U_s = torch.randn(S, R) * 1e-3
        # U_t = torch.randn(T, R) * 5e-3
        U_s_init = normalized_parafac["normalized_spatial_parafac"] # Shape: (S, R)
        U_t_init = normalized_parafac["scaled_temp_parafac"]        # Shape: (T, R)
        assert U_s_init.shape[1] == U_t_init.shape[1] == R

        # k-fold cross-validation
        P_indices = np.arange(P)
        np.random.seed(1234)
        np.random.shuffle(P_indices)
        k_folds = 10
        folds   = np.array_split(P_indices, k_folds)

        mse_kfold_train_lst = []
        mse_kfold_test_lst  = []
        for i_folds in range(k_folds):
            test_idx  = folds[i_folds]
            train_idx = np.setdiff1d(P_indices, test_idx)

            X_train = X_observed[:, :, train_idx]
            X_test  = X_observed[:, :, test_idx]

            ftd_matrices, mse_als = ftd.fit(X_train, Phi[train_idx, :], U_s_init, U_t_init, R)
            mse_kfold_train_lst.append(mse_als)

            U_s, U_t, C = ftd_matrices["U_s"], ftd_matrices["U_t"], ftd_matrices["C"]

            # X_pred       = torch.einsum('sr,tr,pr->stp', U_s, U_t, (Phi[test_idx, :] @ C))
            # mse_als_test = torch.nn.functional.mse_loss(X_pred, X_test) / torch.mean(X_test**2)

            X_pred       = np.einsum('sr,tr,pr->stp', U_s, U_t, Phi[test_idx, :] @ C)
            mse_als_test = np.mean((X_pred - X_test) ** 2) / np.mean(X_test**2)
            
            mse_kfold_test_lst.append(mse_als_test)

        results_als_combinations[tuple(variables_ftd_combination)]['train_mse'].append(mse_kfold_train_lst) # error in each fold
        results_als_combinations[tuple(variables_ftd_combination)]['test_mse'].append(mse_kfold_test_lst) # error in each fold
    
    print ("\n")

In [ ]:
font_size = 14
n_combinations = len(results_als_combinations)
n_cols_mse = 3
n_rows_mse = (n_combinations + n_cols_mse - 1) // n_cols_mse

# Setup the figure
fig, axes = plt.subplots(n_rows_mse, n_cols_mse, figsize=(5*n_cols_mse, 4*n_rows_mse), sharey=True, layout='constrained')
axs = axes.flatten()
colors = plt.get_cmap('tab10').colors#[4:]
r_values = np.array(R_lst)

for i, (combo_name, combination_mse) in enumerate(results_als_combinations.items()):
    ax = axs[i]
    color = colors[i % len(colors)]
    label_str = ", ".join(combo_name)
    
    ax.plot(r_values, mse_parafac_lst, color='black', linestyle='-', label='CP', alpha=0.5, zorder=1)

    # Train error
    train_means = np.mean(combination_mse['train_mse'], axis=1)
    ax.plot(r_values, train_means, marker='o', markersize=4, color=color, label='Train Mean', zorder=4)

    # Test error
    test_data = np.array(combination_mse['test_mse']) # Shape: (Ranks, Simulations)
    test_means = np.mean(test_data, axis=1)
    
    # Plot individual test points
    for r_idx, r_val in enumerate(r_values):
        errors = test_data[r_idx, :]
        jitter = np.random.uniform(-0.0, 0.0, size=len(errors))
        ax.scatter(r_val + jitter, errors, marker='x', color=color, s=20, alpha=0.4, linewidths=0.5)

    ax.plot(r_values, test_means, linestyle='--', color=color, label='Test Mean', zorder=5)
    
    ax.fill_between(r_values, np.min(test_data, axis=1), np.max(test_data, axis=1), 
                    color=color, alpha=0.15, label='Test Range', zorder=2)

    label_str = label_str.replace("p_transported_concatenated_norm", r"$p_{\text{norm}}$")
    label_str = label_str.replace("DeltaV_V", r"$\Delta V / V_{\rm{exhal}}$")
    label_str = label_str.replace("p_end_inh_mean", r"$p_{\rm{i}}$")

    ax.set_title(f"Variables: {label_str}", size=font_size)
    ax.set_yscale('log')
    ax.set_xlabel('Rank', size=font_size)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_xticks(r_values)
    ax.tick_params(axis="both", labelsize=font_size)
    
    if i%n_cols_mse == 0:
        ax.set_ylabel(r'$\mathrm{NMSE}_{\mathrm{ROM}}$', size=font_size)
    
    # Refined legend to include the 'x' marker indicator
    test_points_handle = Line2D([0], [0], marker='x', color=color, linestyle='None', label='Test Elements')
    handles, labels = ax.get_legend_handles_labels()
    handles.append(test_points_handle)
    ax.legend(handles=handles, fontsize=font_size-2, loc='upper right')

    ax.set_ylim(0.1, 2.)
    # ax.tick_params(labelsize=font_size)

for j in range(i + 1, len(axs)):
    fig.delaxes(axs[j])

plt.savefig(f"Results_reduced_model/{Simulations_filename}/FunctionalTensorDecomposition/ftd_mse_train_test_filled.pdf", bbox_inches='tight')
plt.show()

#### FTD with the selected combination of variables and rank

In [ ]:
R_selected = 2
variables_combo_selected = variables_ftd_combinations[8]
print ("Selected combination for final model:", variables_combo_selected)

ftd = FTD(variables_combo_selected, var_dict, K)
Phi, nodes_dict = ftd.build_global_design_matrix()

normalized_parafac, mse_parafac = CP().get_normalized_parafac_decomposition(data_tensor=P_3d_ftd, rank=R_selected)

U_s_init = normalized_parafac["normalized_spatial_parafac"] # (S, R)
U_t_init = normalized_parafac["scaled_temp_parafac"]        # (T, R)
assert U_s_init.shape[1] == U_t_init.shape[1] == R_selected

ftd_matrices, ftd_mse = ftd.fit(X_observed, Phi, U_s_init, U_t_init, R_selected)
print (f"Final ALS Loss on full data with R={R_selected}: {ftd_mse:.6f}")

U_s, U_t, C = ftd_matrices["U_s"], ftd_matrices["U_t"], ftd_matrices["C"]

##### Save CP results, as reference

In [ ]:
# Save CP modes
ind_ini = 0
for region in region_lst:
    with dolfin.XDMFFile(f"Results_reduced_model/{Simulations_filename}/FunctionalTensorDecomposition/CP_reference/cp_modes_target_{acquisition_ref}_{vol_ID_ref[-2:]}_{region}.xdmf") as xdmf:
        xdmf.parameters["flush_output"] = True
        xdmf.parameters["functions_share_mesh"] = True
        xdmf.parameters["rewrite_function_mesh"] = False
        

        for ind_r in range(R_selected):
            ftd_mode_fn = dolfin.Function(fs_DG0_regions_bmesh_ref[region])
            ftd_mode_fn.rename(f"cp_{ind_r+1:02d}", f"cp_{ind_r+1:02d}")

            g_r = normalized_parafac["scaled_temp_parafac"][:, ind_r].reshape(-1,1) @ normalized_parafac["normalized_spatial_parafac"][ind_ini:ind_ini+fs_DG0_regions_bmesh_ref[region].dim(), ind_r].reshape(1,-1)

            for phase_i in range(n_phases):
                ftd_mode_fn.vector()[:] = g_r[phase_i, :]
                xdmf.write(ftd_mode_fn, phase_i)
    ind_ini += fs_DG0_regions_bmesh_ref[region].dim()

In [ ]:
font_size = 14

# Plot temporal modes
plt.figure()
for ind_r in range(R_selected):
    plt.plot(normalized_parafac["scaled_temp_parafac"][:, ind_r], label=f'Mode {ind_r+1}')

plt.xlabel('Phase', fontsize=font_size)
plt.ylabel('Temporal Modes', fontsize=font_size)
plt.tick_params(axis='both', labelsize=font_size)
plt.legend(fontsize=font_size)

plt.savefig(f"Results_reduced_model/{Simulations_filename}/FunctionalTensorDecomposition/CP_reference/cp_reference_temporal_modes.pdf", bbox_inches='tight')
plt.show()

In [ ]:
plt.figure()

cmap, norm = symm_cmap_and_norm(normalized_parafac["normalized_pat_parafac"])

im = plt.imshow(normalized_parafac["normalized_pat_parafac"], aspect='auto', cmap=cmap, interpolation='nearest', norm=norm)

# plt.title(rf'CP acq. component')
plt.xlabel('Mode', fontsize=font_size)
plt.ylabel('Acquisitions', fontsize=font_size)

total_acq = normalized_parafac["normalized_pat_parafac"].shape[0]

pos_supine = total_acq / 4
pos_prone  = 3 * total_acq / 4

plt.yticks([pos_supine, pos_prone], ['Supine', 'Prone'])
plt.xticks(range(R_selected), labels=range(1, R_selected + 1))

plt.tick_params(axis='y', which='both', left=False, labelsize=font_size)
plt.tick_params(axis='x', labelsize=font_size)

cbar = plt.colorbar(im, pad=0.05, shrink=0.8)
cbar.ax.tick_params(labelsize=font_size)

plt.gca().xaxis.set_major_locator(plt.MaxNLocator(integer=True))
# plt.gca().yaxis.set_major_locator(plt.MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig(f"Results_reduced_model/{Simulations_filename}/FunctionalTensorDecomposition/CP_reference/cp_reference_acq_modes.pdf", bbox_inches='tight')
plt.show()

##### Results of FTD wich selected variables

In [ ]:
# Save FTD modes
ind_ini = 0
for region in region_lst:
    with dolfin.XDMFFile(f"Results_reduced_model/{Simulations_filename}/FunctionalTensorDecomposition/ftd_modes_target_{acquisition_ref}_{vol_ID_ref[-2:]}_{region}.xdmf") as xdmf:
        xdmf.parameters["flush_output"] = True
        xdmf.parameters["functions_share_mesh"] = True
        xdmf.parameters["rewrite_function_mesh"] = False
        

        for ind_r in range(R_selected):
            ftd_mode_fn = dolfin.Function(fs_DG0_regions_bmesh_ref[region])
            ftd_mode_fn.rename(f"ftd_{ind_r+1:02d}", f"ftd_{ind_r+1:02d}")

            g_r = U_t[:, ind_r].reshape(-1,1) @ U_s[ind_ini:ind_ini+fs_DG0_regions_bmesh_ref[region].dim(), ind_r].reshape(1,-1)

            for phase_i in range(n_phases):
                ftd_mode_fn.vector()[:] = g_r[phase_i, :]
                xdmf.write(ftd_mode_fn, phase_i)
    ind_ini += fs_DG0_regions_bmesh_ref[region].dim()

In [ ]:
Figures_FTD = FiguresFTD(K, var_dict, U_s, U_t, Phi, C, variables_combo_selected, nodes_dict)
Figures_FTD.plot_temporal_modes(filename=f"Results_reduced_model/{Simulations_filename}/FunctionalTensorDecomposition/ftd_temporal_modes.pdf")

In [ ]:
Figures_FTD.plot_variables_modes(filename=f"Results_reduced_model/{Simulations_filename}/FunctionalTensorDecomposition/ftd_C_coeffs.pdf")

### FTD - Predicting fields

##### Computing residuals

In [ ]:
res_sampling = ResidualSampling(X_observed, U_s, U_t, Phi, C, variables_combo_selected, nodes_dict)

print ("Residual Sampling Mean and Covariance Matrix:")
print (res_sampling.mvn.mean)
print (res_sampling.mvn.covariance_matrix)

##### Creating new pleural pressure fields

In [ ]:
ID_comp  = '230728_02CL05'
acq_comp = 'ANTE_SUP2'
# acq_comp = 'ANTE_PRO1'

vars_comp = reduced_model[(acq_comp, ID_comp)]

if acq_comp == "ANTE_SUP2":
    pos_comp = 0
    torch.manual_seed(123)
elif acq_comp == "ANTE_PRO1":
    pos_comp = 1
    torch.manual_seed(456)

p_norm_comp    = vars_comp['p_transported_concatenated_norm']
DV_V_comp      = vars_comp['DeltaV_V']
p_end_inh_comp = vars_comp['p_end_inh_mean']

In [ ]:
var_dict_new = {
    'Position': [pos_comp],
    'p_transported_concatenated_norm': [p_norm_comp], # TODO Select other values
    'DeltaV_V': [DV_V_comp],
    'p_end_inh_mean': [p_end_inh_comp]}

Phi_new   = res_sampling.get_Phi_new(var_dict_new)

n_sampled = 10

P_f_syn_sampled_lst = []
for i_sampled in range(n_sampled):
    P_syn   = res_sampling.generate_synthetic_field(Phi_new)
    P_f_syn_sampled_lst.append(P_syn) # matrix with n_sampled fields n_samples x n_space x n_phases

P_f_syn = np.stack(P_f_syn_sampled_lst, axis=0) # n_sampled x n_space x n_phases

In [ ]:
# Transport to original geometry
phase_end_exhal = 0
for region in region_lst:
    # Find "inverse" transport plan, to tranport field from reference geometry to the original geometry of the selected volunteer
    for simulation_i in results_volunteers.values():
        if simulation_i['acquisition'] == acq_comp and simulation_i['ID'] == ID_comp and simulation_i['region'] == region:
            alpha_comp   = simulation_i['alpha']
            gravity_comp = simulation_i['gravity']
            pe_comp      = simulation_i['pe']

    with open(f"TransportPlansPOT/{Simulations_filename}/"
            f"target_{acquisition_ref}_{vol_ID_ref[-2:]}_{region}_"
            f"source_{acq_comp}_{ID_comp[-2:]}_{region}.pkl", "rb") as f:
        data        = pickle.load(f)
        T_loaded    = data["T"].toarray()
        log_loaded  = data["log"]
        info_loaded = data["info"]

    if region == "LL":
        ind_ini = 0
    elif region == "RL":
        ind_ini = fs_DG0_regions_bmesh_ref['LL'].dim()

    P_estimation_lst = []
    for i_sampled in range(n_sampled):
        P_syn = P_f_syn[i_sampled, :, :]
        assert fs_DG0_regions_bmesh_ref['LL'].dim() + fs_DG0_regions_bmesh_ref['RL'].dim() == P_syn.shape[0], "Dimension mismatch"


        P_syn_backtransported = np.zeros((T_loaded.shape[0], n_phases))
        for phase_i in range(n_phases):
            P_syn_region            = P_syn[ind_ini:ind_ini+fs_DG0_regions_bmesh_ref[region].dim(), phase_i]
            P_syn_backtransported_i = np.matmul(T_loaded, P_syn_region.reshape(-1,1)) / T_loaded.sum(axis=1, keepdims=True)
            P_syn_backtransported[:,phase_i] = P_syn_backtransported_i.reshape(1,-1)


        # Read end-exhalation mesh
        resultsPath_comp = f"./Results_{acq_comp}/{ID_comp}"
        mesh_unloaded    = dolfin.Mesh()
        with dolfin.XDMFFile(f"{resultsPath_comp}/Pleural_pressure_estimation/"
                            f"mesh_unloaded_{region}_alpha{alpha_comp}_gravity{gravity_comp}_pe{pe_comp}.xdmf") as xdmf_in:
            xdmf_in.read(mesh_unloaded)
        bmesh_unloaded = dolfin.BoundaryMesh(mesh_unloaded, "exterior")
        
        # Read pressure values
        fs_DG0_comp = dolfin.FunctionSpace(bmesh_unloaded, "DG", 0)


        vfs_CG1 = dolfin.VectorFunctionSpace(bmesh_unloaded, "CG", 1)
        U_b   = dolfin.Function(vfs_CG1)

        MDR = MeshDisplacementReader(acq_comp, ID_comp, region, n_phases)

        # Read center of mass
        fs_R           = dolfin.VectorFunctionSpace(bmesh_unloaded, "R", 0)
        center_of_mass = dolfin.Function(fs_R)

        h5_results = dolfin.HDF5File(bmesh_unloaded.mpi_comm(), f"{resultsPath_comp}/Pleural_pressure_estimation/data_{region}_alpha{alpha_comp}_gravity{gravity_comp}_pe{pe_comp}.h5", "r")


        p_estimation = np.zeros((bmesh_unloaded.num_cells(), n_phases))

        for phase_i in range(n_phases):
            U_b_name = "/U_b/vector_%d"%phase_i
            h5_results.read(U_b, U_b_name)
            bmesh_deformed_phase_i = dolfin.BoundaryMesh(mesh_unloaded, "exterior")
            dolfin.ALE.move(bmesh_deformed_phase_i, U_b)
            
            fs_DG0_phase_i         = dolfin.FunctionSpace(bmesh_deformed_phase_i, "DG", 0)
            def_cells_coords       = fs_DG0_phase_i.tabulate_dof_coordinates()

            cent_name = "/center_of_mass/vector_%d"%phase_i
            h5_results.read(center_of_mass, cent_name)
            

            x_tilde       = def_cells_coords - center_of_mass.vector().get_local().reshape(1,-1)
            p_syn_phase_i = pe_comp + rho_solid * x_tilde.dot(np.array([gravity_comp*g, 0, 0])) + P_syn_backtransported[:, phase_i]
            p_estimation[:, phase_i] = p_syn_phase_i

        P_estimation_lst.append(p_estimation)

        
    ## Save P in original geometry
    p_est_fn = dolfin.Function(fs_DG0_comp)
    with dolfin.XDMFFile(f"Results_reduced_model/{Simulations_filename}/FunctionalTensorDecomposition/Sampling_new_fields/Sampled_P_{ID_comp[-2:]}_{acq_comp}_{region}.xdmf") as xdmf:
        xdmf.parameters["flush_output"] = True
        xdmf.parameters["functions_share_mesh"] = True
        xdmf.parameters["rewrite_function_mesh"] = False

        for i_sampled in range(n_sampled):
            p_est_fn.rename(f"p_syn{i_sampled}", f"p_syn{i_sampled}")
            for phase_i in range(n_phases):
                p_est_fn.vector()[:] = P_estimation_lst[i_sampled][:, phase_i].ravel()
                xdmf.write(p_est_fn, phase_i)
        

##### Error vs original field

In [ ]:
for n_sim, simulation_i in results_volunteers.items():
    
    if simulation_i["ID"] == vol_ID_ref and simulation_i["acquisition"] == acquisition_ref:
        p_org_pat = simulation_i["p"]

        if simulation_i["region"] == "LL":
            P_ref_LL = p_org_pat

        elif simulation_i["region"] == "RL":
            P_ref_RL = p_org_pat

P_ref_list = []
for region in region_lst:
    P_ref_list.append(P_ref_LL if region == "LL" else P_ref_RL)

P_ref = np.concatenate(P_ref_list, axis=1)

In [ ]:
font_size = 12
n_rows_err_sampled = 1
n_cols_err_sampled = 1

fig, axes = plt.subplots(n_rows_err_sampled, n_cols_err_sampled, figsize=(5 * n_cols_err_sampled, 4 * n_rows_err_sampled), sharey=True)
# axes = axes.flatten()

P_f_reference = reduced_model[acq_comp, ID_comp]['p_transported_concatenated']

for i_sampled in range(n_sampled):
    P_f_i = P_f_syn[i_sampled,:,:].T

    err_phases = np.zeros(n_phases)
    for phase_i in range(n_phases):
        err_phases[phase_i] = np.linalg.norm((P_f_reference[phase_i,:] - P_f_i[phase_i,:]))**2 / np.linalg.norm(P_ref[phase_i,:])**2

    axes.plot(err_phases) 

axes.set_xlabel('Phase', fontsize=font_size)
axes.set_ylabel(r"$\mathrm{NMSE}_{\mathrm{pred}}$", fontsize=font_size)
axes.tick_params(axis="both", labelsize=font_size)

axes.set_xlim([0,31])
axes.set_ylim([0,0.1])

# axes.set_yticks(np.arange(12, 32, 3))

plt.tight_layout()
plt.savefig(f"Results_reduced_model/{Simulations_filename}/FunctionalTensorDecomposition/Sampling_new_fields/error_sampled_fields_{acq_comp}.pdf", bbox_inches='tight')
plt.show()